# PH-SHOWOA GPU benchmark: RCdp207 (pop_size=30, max_iter=1000, runs=30)


In [ ]:
INSTANCE = 'RCdp207'
JOB_ID = 'd5672ddf9777'
RUNS = 30
MAX_ITER = 1000
POP_SIZE = 30
SEED0 = 42
GIT_URL = 'https://github.com/Welkie/ph-showoa.git'
GIT_COMMIT = 'f33b63fa7da189ec91109813e896985261226889'

In [ ]:
import os, subprocess, sys
REPO = "/tmp/ph-showoa"
if not os.path.isdir(os.path.join(REPO, "src_python_gpu_SA_RCRS_GRASP")):
    subprocess.run(["git", "init", "-q", REPO], check=True)
    subprocess.run(["git", "-C", REPO, "remote", "add", "origin", GIT_URL], check=True)
    if subprocess.run(["git", "-C", REPO, "fetch", "-q", "--depth", "1", "origin", GIT_COMMIT]).returncode != 0:
        subprocess.run(["git", "-C", REPO, "fetch", "-q", "origin"], check=True)
        subprocess.run(["git", "-C", REPO, "checkout", "-q", GIT_COMMIT], check=True)
    else:
        subprocess.run(["git", "-C", REPO, "checkout", "-q", "FETCH_HEAD"], check=True)
head = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("Code commit:", head)
assert head == GIT_COMMIT, "unexpected commit checked out"
subprocess.run(["nvidia-smi"])
from numba import cuda
import torch
if not (cuda.is_available() and torch.cuda.is_available()):
    raise RuntimeError("NO_GPU: CUDA GPU is not available in this session (GPU quota exhausted or accelerator disabled)")
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import csv, glob, json, os, re, subprocess, sys, time
OUT = "/kaggle/working"
LOGS = os.path.join(OUT, "logs")
os.makedirs(LOGS, exist_ok=True)

def find_problem(name):
    fn = f"explicit_{name.lower()}.vrpsdptw"
    hits = glob.glob(f"/kaggle/input/**/Wang_Chen/{fn}", recursive=True) or glob.glob(f"/kaggle/input/**/{fn}", recursive=True)
    if not hits:
        local = os.path.join(REPO, "dataset", fn)
        hits = [local] if os.path.isfile(local) else []
    if not hits:
        raise FileNotFoundError(fn)
    return hits[0]

PROBLEM = find_problem(INSTANCE)
print("Problem file:", PROBLEM)

def run_solver(seed, tag):
    # Same timing as the src_python_baseline runner: one fresh process per run, timed from
    # launch to exit (Python start-up, imports, data loading, Numba JIT compile and the solve).
    cmd = [sys.executable, "-m", "src_python_gpu_SA_RCRS_GRASP.main", "--problem", PROBLEM,
           "--compute_backend", "cuda", "--init", "sa_rcrs_grasp", "--paper_flags",
           "--architecture", "python_cuda", "--runs", "1", "--max_iter", str(MAX_ITER),
           "--pop_size", str(POP_SIZE), "--workers", "1", "--random_seed", str(seed)]
    log_path = os.path.join(LOGS, f"{tag}.log")
    started = time.perf_counter()
    with open(log_path, "w", encoding="utf-8") as log_fp:
        result = subprocess.run(cmd, cwd=REPO, text=True, stdout=log_fp, stderr=subprocess.STDOUT, check=False)
    dt = time.perf_counter() - started
    out = open(log_path, encoding="utf-8", errors="replace").read()
    m = re.search(r"Run 1 finishes \| Run Best: NV=(\d+), TD=([\d.]+)", out)
    if result.returncode != 0 or not m:
        raise RuntimeError(f"{tag} failed (exit code {result.returncode}):\n" + out[-3000:])
    return int(m.group(1)), float(m.group(2)), dt

rows = []
runs_path = os.path.join(OUT, "runs.csv")
for r in range(1, RUNS + 1):
    seed = SEED0 + r - 1
    nv, td, dt = run_solver(seed, f"run{r:02d}_seed{seed}")
    rows.append({"instance": INSTANCE, "run": r, "seed": seed, "NV": nv, "TD": round(td, 4),
                 "cost": round(2000 * nv + td, 4), "time": round(dt, 3)})
    with open(runs_path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader(); w.writerows(rows)
    print(f"[run {r:2d}/{RUNS}] seed={seed} NV={nv} TD={td:.4f} time={dt:.1f}s", flush=True)

best = min(rows, key=lambda x: (x["NV"], x["TD"]))
summary = {"job_id": JOB_ID, "instance": INSTANCE, "status": "Success", "runs": len(rows),
           "best_NV": best["NV"], "best_TD": best["TD"],
           "avg_time_s": round(sum(x["time"] for x in rows) / len(rows), 3),
           "timing": "per-run fresh process, launch-to-exit wall clock", "commit": GIT_COMMIT}
with open(os.path.join(OUT, "summary.json"), "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
print("SUMMARY:", json.dumps(summary))